# MOFI Tutorial 00: In Silico Simulation Benchmark

This tutorial demonstrates how to use MOFI for single-modal dynamics reconstruction on a controlled simulation dataset where ground truth is fully specified.

The simulation dataset was generated from a regulatory network defining causal relationships among variables X, Y, and I, governed by a state-independent potential landscape Z that contains both attracting and repulsive regions. The system produces two distinct lineages diverging from a common progenitor state.

## 1. Load Packages

In [ ]:
import sys
from pathlib import Path

# Add src to path
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import torch

import CytoBridge.pp as cb_pp
import CytoBridge.tl as cb_tl
from CytoBridge.utils.utils import set_seed

In [ ]:
# Set random seed for reproducibility
set_seed(22)

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 2. Load Data

In [ ]:
# Load simulation data
DATA_PATH = "../datasets/simulation/simulation4_2d.h5ad"
SIMULATION_CSV = "../datasets/simulation/simulation_gene.csv"

adata = sc.read_h5ad(DATA_PATH)
print(f"Data shape: {adata.shape}")
print(f"Obs columns: {list(adata.obs.columns)}")
print(f"Obsm keys: {list(adata.obsm.keys())}")
adata

## 3. Preprocess

Map time points to numeric scale. For simulation data, we use `dim_reduction='none'` since the data is already low-dimensional.

In [ ]:
# Time mapping for simulation: 5 time points
time_mapping = {
    0.0: 0.0,
    0.75: 1.0,
    1.5: 2.0,
    2.25: 3.0,
    3.0: 4.0,
}

adata = cb_pp.preprocess(
    adata,
    time_key='samples',
    time_mapping=time_mapping,
    dim_reduction='none',
    normalization=False,
    log1p=False,
    select_hvg=False
)

print(f"\nTime points: {sorted(adata.obs['time_point_processed'].unique())}")
print(f"Cells per time point:")
print(adata.obs['time_point_processed'].value_counts().sort_index())

## 4. Train Dynamics Model

We use MOFI's Neural ODE + Unbalanced OT framework to learn velocity fields and growth rates from the simulation snapshots.

In [ ]:
# Train the dynamics model
adata = cb_tl.fit(
    adata,
    config='../examples/configs/simu/unbalanced_ot_simu.yaml',
    batch_size=400,
    device=device
)

## 5. Inspect Trained Model

After training, the model outputs are stored in the AnnData object:
- `adata.obsm['velocity_latent']`: Inferred velocity field
- `adata.obsm['growth_rate']`: Inferred growth rates
- `adata.uns['all_model']`: Model weights and config

In [ ]:
print("Model components:", list(adata.uns['all_model']['model_config']['components']))
print(f"Velocity shape: {adata.obsm['velocity_latent'].shape}")
print(f"Growth rate shape: {adata.obsm['growth_rate'].shape}")

## 6. Visualization

Visualize the learned velocity field and growth rates.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot cells colored by time
scatter = axes[0].scatter(
    adata.obsm['X_latent'][:, 0],
    adata.obsm['X_latent'][:, 1],
    c=adata.obs['time_point_processed'],
    cmap='viridis', s=5, alpha=0.6
)
plt.colorbar(scatter, ax=axes[0], label='Time')
axes[0].set_title('Cells colored by time')
axes[0].set_xlabel('X1')
axes[0].set_ylabel('X2')

# Plot velocity field
axes[1].quiver(
    adata.obsm['X_latent'][:, 0],
    adata.obsm['X_latent'][:, 1],
    adata.obsm['velocity_latent'][:, 0],
    adata.obsm['velocity_latent'][:, 1],
    alpha=0.3, scale=20
)
axes[1].set_title('Inferred velocity field')
axes[1].set_xlabel('X1')
axes[1].set_ylabel('X2')

# Plot growth rate
scatter2 = axes[2].scatter(
    adata.obsm['X_latent'][:, 0],
    adata.obsm['X_latent'][:, 1],
    c=adata.obsm['growth_rate'].flatten(),
    cmap='RdBu_r', s=5, alpha=0.6
)
plt.colorbar(scatter2, ax=axes[2], label='Growth rate')
axes[2].set_title('Inferred growth rate')
axes[2].set_xlabel('X1')
axes[2].set_ylabel('X2')

plt.tight_layout()
plt.savefig('../tutorial/save_results/simulation_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

This tutorial demonstrated the basic MOFI workflow:
1. **Load** time-series single-cell data
2. **Preprocess** with time mapping
3. **Train** dynamics model using Neural ODE + Unbalanced OT
4. **Inspect** velocity fields and growth rates
5. **Visualize** the learned dynamics

For dual-modal analysis (RNA + ATAC/Protein), see tutorials 01-03.